<a href="https://colab.research.google.com/github/adit-codez/part-1-neural-network-analysis/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.neural_network import MLPClassifier

matplotlib.use("Agg")
np.random.seed(42)

os.makedirs("results/model_comparison_table.csv", exist_ok=True)

In [27]:
df = pd.read_csv("customer_churn_nn.csv")


def generate_dataset(n: int = 2000, churn_rate: float = 0.032) -> pd.DataFrame:
    """Synthetic customer-churn dataset that mirrors the original statistics."""
    REGIONS   = ["South", "West", "Central", "North", "East"]
    PLANS     = ["Basic", "Standard", "Premium", "Enterprise"]
    CONTRACTS = ["Month-to-month", "One-year", "Two-year"]
    PAYMENTS  = ["Debit Card", "Credit Card", "Wallet", "UPI", "Net Banking"]

    plan_charge_range = {
        "Basic":      (280,  530),
        "Standard":   (500,  900),
        "Premium":    (850, 1350),
        "Enterprise": (1300, 2200),
    }
    plan_data_range = {
        "Basic":        (0.5, 120),
        "Standard":     (20,  145),
        "Premium":      (50,  220),
        "Enterprise":  (150,  270),
    }

    def make_row(churn: int) -> dict:
        region   = np.random.choice(REGIONS)
        plan     = np.random.choice(PLANS,
                       p=[0.20, 0.30, 0.35, 0.15] if churn
                         else [0.35, 0.35, 0.22, 0.08])
        contract = np.random.choice(CONTRACTS,
                       p=[0.85, 0.10, 0.05] if churn
                         else [0.55, 0.28, 0.17])
        tenure   = max(1, int(np.random.exponential(10 if churn else 28)))
        monthly  = round(np.random.uniform(*plan_charge_range[plan]), 2)
        data_gb  = round(np.random.uniform(*plan_data_range[plan]), 2)
        sat      = round(np.random.uniform(1.5, 6.5) if churn
                         else np.random.uniform(4.0, 10.0), 1)
        tickets  = max(0, int(np.random.poisson(3.5 if churn else 1.5)))
        delay    = max(0, int(np.random.exponential(6 if churn else 3)))
        payment  = np.random.choice(PAYMENTS)
        autopay  = np.random.choice([0, 1], p=[0.4, 0.6])
        discount = np.random.choice([0, 5, 10, 15, 20], p=[0.3, 0.2, 0.25, 0.15, 0.1])
        referrals   = max(0, int(np.random.poisson(1)))
        login_days  = max(0, min(30, int(np.random.normal(17, 6))))
        last_complaint = max(0, int(np.random.exponential(60)))
        return {
            "region": region, "plan_type": plan,
            "contract_type": contract, "payment_method": payment,
            "tenure_months": min(72, tenure),
            "monthly_charges_inr": monthly,
            "avg_login_days_per_month": login_days,
            "support_tickets_last_90_days": tickets,
            "payment_delay_days": delay,
            "data_usage_gb": data_gb,
            "satisfaction_score": sat,
            "last_complaint_days_ago": last_complaint,
            "discount_percent": discount,
            "autopay_enabled": autopay,
            "referral_count": referrals,
            "churn": churn,
        }

    n_churn = int(n * churn_rate)
    rows = [make_row(1) for _ in range(n_churn)] + \
           [make_row(0) for _ in range(n - n_churn)]
    np.random.shuffle(rows)
    df = pd.DataFrame(rows)
    df.index = [f"CUST{i+1:04d}" for i in range(n)]
    return df

Task 1 - Dataset Understanding

In [28]:
print("=" * 65)
print("TASK 1 — DATASET UNDERSTANDING")
print("=" * 65)

df = generate_dataset()

print(f"\n[1] Shape          : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n[2] Column types:\n{df.dtypes}")
print(f"\n[3] Missing values :\n{df.isnull().sum()}")

churn_counts = df["churn"].value_counts()
print(f"\n[4] Target variable — churn distribution:")
print(churn_counts)
print(f"    Churn rate     : {churn_counts[1] / len(df) * 100:.1f}%")

print(f"\n[5] Statistical summary:")
print(df.describe().round(2))

TASK 1 — DATASET UNDERSTANDING

[1] Shape          : 2000 rows × 16 columns

[2] Column types:
region                           object
plan_type                        object
contract_type                    object
payment_method                   object
tenure_months                     int64
monthly_charges_inr             float64
avg_login_days_per_month          int64
support_tickets_last_90_days      int64
payment_delay_days                int64
data_usage_gb                   float64
satisfaction_score              float64
last_complaint_days_ago           int64
discount_percent                  int64
autopay_enabled                   int64
referral_count                    int64
churn                             int64
dtype: object

[3] Missing values :
region                          0
plan_type                       0
contract_type                   0
payment_method                  0
tenure_months                   0
monthly_charges_inr             0
avg_login_days_per_month 

Task 2 - Data Preprocessing

In [30]:
print("\n" + "=" * 65)
print("TASK 2 — DATA PREPROCESSING")
print("=" * 65)

# No missing values → nothing to impute

# Encode categorical features
CAT_COLS = ["region", "plan_type", "contract_type", "payment_method"]
le = LabelEncoder()
df_enc = df.copy()
for col in CAT_COLS:
    df_enc[col] = le.fit_transform(df_enc[col])
    print(f"  Encoded '{col}': {sorted(df[col].unique())}")

# Split features / target
X = df_enc.drop("churn", axis=1).values
y = df_enc["churn"].values

# Train / test split (stratified to preserve churn ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standard scaling
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"\n  Train  : {X_train_s.shape}  |  churn rate {y_train.mean()*100:.1f}%")
print(f"  Test   : {X_test_s.shape}   |  churn rate {y_test.mean()*100:.1f}%")


TASK 2 — DATA PREPROCESSING
  Encoded 'region': [np.str_('Central'), np.str_('East'), np.str_('North'), np.str_('South'), np.str_('West')]
  Encoded 'plan_type': [np.str_('Basic'), np.str_('Enterprise'), np.str_('Premium'), np.str_('Standard')]
  Encoded 'contract_type': [np.str_('Month-to-month'), np.str_('One-year'), np.str_('Two-year')]
  Encoded 'payment_method': [np.str_('Credit Card'), np.str_('Debit Card'), np.str_('Net Banking'), np.str_('UPI'), np.str_('Wallet')]

  Train  : (1600, 15)  |  churn rate 3.2%
  Test   : (400, 15)   |  churn rate 3.2%


Task 3 - Neural Network Model

In [31]:
print("\n" + "=" * 65)
print("TASK 3 — NEURAL NETWORK MODEL (built from scratch with NumPy)")
print("=" * 65)


# Activation functions

def relu(z):
    return np.maximum(0.0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1.0 - s)

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-9
    return -np.mean(
        y_true * np.log(y_pred + eps) + (1.0 - y_true) * np.log(1.0 - y_pred + eps)
    )


# NeuralNetwork class

class NeuralNetwork:
    """
    Feed-forward neural network trained with mini-batch gradient descent.

    Parameters
    ----------
    layer_sizes : list[int]
        Number of neurons per layer including input and output.
        Example: [15, 32, 16, 1]
    learning_rate : float
    activation : str   "relu" | "sigmoid"  (hidden layers only)
    """

    def __init__(
        self,
        layer_sizes: list,
        learning_rate: float = 0.005,
        activation: str = "relu",
    ):
        self.lr = learning_rate
        self.act      = relu    if activation == "relu" else sigmoid
        self.act_grad = relu_grad if activation == "relu" else sigmoid_grad
        self.history  = {"loss": [], "val_loss": [], "acc": [], "val_acc": []}

        # He initialisation for ReLU; Xavier-style for Sigmoid
        self.weights = []
        self.biases  = []
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            scale  = np.sqrt(2.0 / fan_in) if activation == "relu" \
                     else np.sqrt(1.0 / fan_in)
            self.weights.append(
                np.random.randn(layer_sizes[i], layer_sizes[i + 1]) * scale
            )
            self.biases.append(np.zeros((1, layer_sizes[i + 1])))

    # Forward pass
    def _forward(self, X: np.ndarray):
        """
        Compute activations for every layer.
        Returns the final output (probabilities).
        Stores intermediate Z and A for backprop.
        """
        self._Z = []          # pre-activations
        self._A = [X]         # post-activations (A[0] = input)

        for i in range(len(self.weights) - 1):
            z = self._A[-1] @ self.weights[i] + self.biases[i]
            self._Z.append(z)
            self._A.append(self.act(z))

        # Output layer always uses sigmoid (binary classification)
        z_out = self._A[-1] @ self.weights[-1] + self.biases[-1]
        self._Z.append(z_out)
        out = sigmoid(z_out)
        self._A.append(out)
        return out

    # Backward pass (backpropagation)
    def _backward(self, y: np.ndarray):
        """
        Compute gradients via chain rule and update weights in place.
        """
        m = y.shape[0]

        # Gradient of BCE loss w.r.t. output activation
        y_col = y.reshape(-1, 1)
        dA = -(
            y_col / np.clip(self._A[-1], 1e-9, None)
            - (1 - y_col) / np.clip(1 - self._A[-1], 1e-9, None)
        )

        for i in reversed(range(len(self.weights))):

            if i == len(self.weights) - 1:
                dZ = dA * sigmoid_grad(self._Z[i])
            else:
                dZ = dA * self.act_grad(self._Z[i])

            dW = self._A[i].T @ dZ / m
            db = dZ.mean(axis=0, keepdims=True)

            # Gradient for the previous layer
            dA = dZ @ self.weights[i].T

            # Gradient descent update
            self.weights[i] -= self.lr * dW
            self.biases[i]  -= self.lr * db

    # Training loop
    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        epochs: int = 100,
        batch_size: int = 64,
        verbose: bool = True,
    ):
        for epoch in range(1, epochs + 1):
            # Shuffle training data each epoch
            idx = np.random.permutation(len(X_train))

            for start in range(0, len(X_train), batch_size):
                batch_idx = idx[start : start + batch_size]
                self._forward(X_train[batch_idx])
                self._backward(y_train[batch_idx])

            # Record metrics
            tr_pred = self._forward(X_train).flatten()
            vl_pred = self._forward(X_val).flatten()

            tr_loss = binary_cross_entropy(y_train, tr_pred)
            vl_loss = binary_cross_entropy(y_val,   vl_pred)
            tr_acc  = ((tr_pred > 0.5) == y_train).mean()
            vl_acc  = ((vl_pred > 0.5) == y_val).mean()

            self.history["loss"].append(tr_loss)
            self.history["val_loss"].append(vl_loss)
            self.history["acc"].append(tr_acc)
            self.history["val_acc"].append(vl_acc)

            if verbose and epoch % 20 == 0:
                print(
                    f"  Epoch {epoch:3d}/{epochs}"
                    f"  loss={tr_loss:.4f}  val_loss={vl_loss:.4f}"
                    f"  acc={tr_acc:.4f}  val_acc={vl_acc:.4f}"
                )

    # Inference
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self._forward(X).flatten()

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) > threshold).astype(int)


# Instantiate and train

n_features = X_train_s.shape[1]

model = NeuralNetwork(
    layer_sizes    = [n_features, 32, 16, 1],
    learning_rate  = 0.005,
    activation     = "relu",
)

print(f"\n  Architecture : [{n_features}] → [32] → [16] → [1]")
print(f"  Activation   : ReLU (hidden)  |  Sigmoid (output)")
print(f"  Loss         : Binary Cross-Entropy")
print(f"  LR           : 0.005  |  Batch size : 64  |  Epochs : 100\n")

model.fit(
    X_train_s, y_train,
    X_test_s,  y_test,
    epochs=100, batch_size=64, verbose=True,
)


TASK 3 — NEURAL NETWORK MODEL (built from scratch with NumPy)

  Architecture : [15] → [32] → [16] → [1]
  Activation   : ReLU (hidden)  |  Sigmoid (output)
  Loss         : Binary Cross-Entropy
  LR           : 0.005  |  Batch size : 64  |  Epochs : 100

  Epoch  20/100  loss=0.1609  val_loss=0.1633  acc=0.9681  val_acc=0.9675
  Epoch  40/100  loss=0.1337  val_loss=0.1413  acc=0.9681  val_acc=0.9675
  Epoch  60/100  loss=0.1171  val_loss=0.1294  acc=0.9681  val_acc=0.9675
  Epoch  80/100  loss=0.1053  val_loss=0.1211  acc=0.9681  val_acc=0.9675
  Epoch 100/100  loss=0.0964  val_loss=0.1151  acc=0.9681  val_acc=0.9675


Task 4 - Training Evaluation

In [32]:
print("\n" + "=" * 65)
print("TASK 4 — TRAINING AND EVALUATION")
print("=" * 65)

y_pred = model.predict(X_test_s)

print(f"\n  Final train accuracy : {model.history['acc'][-1]*100:.2f}%")
print(f"  Final test  accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"\n  Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))


# Figure 1 — EDA plots

fig1, axes = plt.subplots(2, 3, figsize=(15, 9))
fig1.suptitle("Task 1 — Exploratory Data Analysis", fontsize=16, fontweight="bold")

# 1a — churn distribution
vc = df["churn"].value_counts()
bars = axes[0, 0].bar(["No Churn", "Churn"], vc.values,
                       color=["#2563EB", "#DC2626"], edgecolor="white")
for b in bars:
    axes[0, 0].text(b.get_x() + b.get_width() / 2, b.get_height() + 5,
                    str(b.get_height()), ha="center", fontweight="bold")
axes[0, 0].set_title("Target Variable Distribution", fontweight="bold")
axes[0, 0].set_ylabel("Count")

# 1b — tenure by churn
df.boxplot(column="tenure_months", by="churn", ax=axes[0, 1],
           patch_artist=True,
           boxprops=dict(facecolor="#BFDBFE"),
           medianprops=dict(color="#DC2626", linewidth=2))
axes[0, 1].set_title("Tenure by Churn", fontweight="bold")
axes[0, 1].set_xlabel("Churn (0=No, 1=Yes)")
plt.sca(axes[0, 1]); plt.title("")

# 1c — satisfaction score by churn
df.boxplot(column="satisfaction_score", by="churn", ax=axes[0, 2],
           patch_artist=True,
           boxprops=dict(facecolor="#BBF7D0"),
           medianprops=dict(color="#DC2626", linewidth=2))
axes[0, 2].set_title("Satisfaction Score by Churn", fontweight="bold")
axes[0, 2].set_xlabel("Churn (0=No, 1=Yes)")
plt.sca(axes[0, 2]); plt.title("")

# 1d — contract type
ct = df.groupby(["contract_type", "churn"]).size().unstack(fill_value=0)
ct.plot(kind="bar", ax=axes[1, 0], color=["#2563EB", "#DC2626"], edgecolor="white")
axes[1, 0].set_title("Contract Type vs Churn", fontweight="bold")
axes[1, 0].tick_params(axis="x", rotation=20)
axes[1, 0].legend(["No Churn", "Churn"])
axes[1, 0].set_xlabel("")

# 1e — monthly charges
axes[1, 1].hist(df[df.churn == 0]["monthly_charges_inr"], bins=30,
                alpha=0.7, color="#2563EB", label="No Churn")
axes[1, 1].hist(df[df.churn == 1]["monthly_charges_inr"], bins=30,
                alpha=0.7, color="#DC2626", label="Churn")
axes[1, 1].set_title("Monthly Charges Distribution", fontweight="bold")
axes[1, 1].set_xlabel("Monthly Charges (INR)")
axes[1, 1].legend()

# 1f — support tickets
tk = df.groupby(["support_tickets_last_90_days", "churn"]).size().unstack(fill_value=0)
tk.plot(kind="bar", ax=axes[1, 2], color=["#2563EB", "#DC2626"], edgecolor="white")
axes[1, 2].set_title("Support Tickets vs Churn", fontweight="bold")
axes[1, 2].tick_params(axis="x", rotation=0)
axes[1, 2].legend(["No Churn", "Churn"])
axes[1, 2].set_xlabel("Tickets (last 90 days)")

plt.tight_layout()
fig1.savefig("results/fig1_eda.png", dpi=150, bbox_inches="tight")
plt.close(fig1)
print("\n  Saved → results/fig1_eda.png")


# Figure 2 — Training curves

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle("Task 4 — Training & Validation Curves (Custom NumPy NN)",
              fontsize=14, fontweight="bold")
epochs_range = range(1, len(model.history["loss"]) + 1)

axes2[0].plot(epochs_range, model.history["loss"],     color="#2563EB", lw=2, label="Train Loss")
axes2[0].plot(epochs_range, model.history["val_loss"], color="#DC2626", lw=2, ls="--", label="Val Loss")
axes2[0].set_title("Loss over Epochs", fontweight="bold")
axes2[0].set_xlabel("Epoch"); axes2[0].set_ylabel("BCE Loss"); axes2[0].legend()

axes2[1].plot(epochs_range, [a * 100 for a in model.history["acc"]],
              color="#16A34A", lw=2, label="Train Acc")
axes2[1].plot(epochs_range, [a * 100 for a in model.history["val_acc"]],
              color="#EA580C", lw=2, ls="--", label="Val Acc")
axes2[1].set_title("Accuracy over Epochs", fontweight="bold")
axes2[1].set_xlabel("Epoch"); axes2[1].set_ylabel("Accuracy (%)"); axes2[1].legend()

plt.tight_layout()
fig2.savefig("results/fig2_training_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig2)
print("  Saved → results/fig2_training_curves.png")


# Figure 3 — Confusion matrix

fig3, ax3 = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax3,
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"],
            linewidths=1, linecolor="white")
ax3.set_title("Confusion Matrix — Custom NN (Test Set)", fontweight="bold")
ax3.set_ylabel("Actual"); ax3.set_xlabel("Predicted")
plt.tight_layout()
fig3.savefig("results/fig3_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close(fig3)
print("  Saved → results/fig3_confusion_matrix.png")



TASK 4 — TRAINING AND EVALUATION

  Final train accuracy : 96.81%
  Final test  accuracy : 96.75%

  Classification Report:

              precision    recall  f1-score   support

    No Churn       0.97      1.00      0.98       387
       Churn       0.00      0.00      0.00        13

    accuracy                           0.97       400
   macro avg       0.48      0.50      0.49       400
weighted avg       0.94      0.97      0.95       400



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



  Saved → results/fig1_eda.png
  Saved → results/fig2_training_curves.png
  Saved → results/fig3_confusion_matrix.png


Task 5 - Hyperparameter Experiments

In [33]:
print("\n" + "=" * 65)
print("TASK 5 — HYPERPARAMETER EXPERIMENTS")
print("=" * 65)

EXPERIMENTS = [
    {
        "name": "Exp 1 — Baseline",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "learning_rate_init": 0.005,
        "batch_size": 64,
        "max_iter": 100,
    },
    {
        "name": "Exp 2 — Deeper Network",
        "hidden_layer_sizes": (64, 32, 16),
        "activation": "relu",
        "learning_rate_init": 0.005,
        "batch_size": 64,
        "max_iter": 100,
    },
    {
        "name": "Exp 3 — High Learning Rate",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "learning_rate_init": 0.10,
        "batch_size": 64,
        "max_iter": 100,
    },
    {
        "name": "Exp 4 — Low Learning Rate",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "learning_rate_init": 0.0005,
        "batch_size": 64,
        "max_iter": 100,
    },
    {
        "name": "Exp 5 — Sigmoid Activation",
        "hidden_layer_sizes": (32, 16),
        "activation": "logistic",
        "learning_rate_init": 0.005,
        "batch_size": 64,
        "max_iter": 100,
    },
    {
        "name": "Exp 6 — Large Batch Size",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "learning_rate_init": 0.005,
        "batch_size": 256,
        "max_iter": 100,
    },
    {
        "name": "Exp 7 — Wider Network",
        "hidden_layer_sizes": (128, 64),
        "activation": "relu",
        "learning_rate_init": 0.005,
        "batch_size": 64,
        "max_iter": 100,
    },
]

results = []
for exp in EXPERIMENTS:
    kwargs = {k: v for k, v in exp.items() if k != "name"}
    kwargs.update({"solver": "adam", "random_state": 42, "early_stopping": False})

    clf = MLPClassifier(**kwargs)
    clf.fit(X_train_s, y_train)

    tr_acc = accuracy_score(y_train, clf.predict(X_train_s))
    te_acc = accuracy_score(y_test,  clf.predict(X_test_s))

    results.append({
        "Experiment":    exp["name"],
        "Architecture":  str(exp["hidden_layer_sizes"]),
        "Activation":    exp["activation"],
        "LR":            exp["learning_rate_init"],
        "Batch Size":    exp["batch_size"],
        "Train Acc (%)": round(tr_acc * 100, 2),
        "Test Acc (%)":  round(te_acc * 100, 2),
        "Final Loss":    round(clf.loss_, 4),
    })
    print(
        f"  {exp['name']:<30} "
        f"train={tr_acc*100:.2f}%  test={te_acc*100:.2f}%  loss={clf.loss_:.4f}"
    )

results_df = pd.DataFrame(results)
print(f"\n{'='*65}")
print(results_df.to_string(index=False))

results_df.to_csv("results/experiment_results.csv", index=False)
print("\n  Saved → results/experiment_results.csv")


# Figure 4 — Experiment comparison bar chart

fig4, ax4 = plt.subplots(figsize=(14, 5))
exp_names  = results_df["Experiment"].tolist()
train_accs = results_df["Train Acc (%)"].tolist()
test_accs  = results_df["Test Acc (%)"].tolist()

x = np.arange(len(exp_names))
w = 0.35
bars_tr = ax4.bar(x - w / 2, train_accs, w, color="#2563EB",
                  edgecolor="white", label="Train Acc")
bars_te = ax4.bar(x + w / 2, test_accs,  w, color="#DC2626",
                  edgecolor="white", label="Test Acc")

for b in list(bars_tr) + list(bars_te):
    ax4.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.1,
             f"{b.get_height():.1f}%", ha="center", va="bottom",
             fontsize=8.5, fontweight="bold")

ax4.set_xticks(x)
ax4.set_xticklabels(exp_names, rotation=18, ha="right")
ax4.set_title("Task 5 — Hyperparameter Experiment Comparison",
              fontweight="bold", fontsize=13)
ax4.set_ylabel("Accuracy (%)")
ax4.set_ylim(85, 104)
ax4.legend()
plt.tight_layout()
fig4.savefig("results/fig4_experiment_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig4)
print("  Saved → results/fig4_experiment_comparison.png")



TASK 5 — HYPERPARAMETER EXPERIMENTS
  Exp 1 — Baseline               train=100.00%  test=98.25%  loss=0.0005
  Exp 2 — Deeper Network         train=100.00%  test=98.00%  loss=0.0002
  Exp 3 — High Learning Rate     train=96.81%  test=96.75%  loss=0.0851


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


  Exp 4 — Low Learning Rate      train=99.19%  test=98.50%  loss=0.0248


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


  Exp 5 — Sigmoid Activation     train=99.56%  test=98.00%  loss=0.0220


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


  Exp 6 — Large Batch Size       train=100.00%  test=98.25%  loss=0.0017
  Exp 7 — Wider Network          train=100.00%  test=98.00%  loss=0.0003

                Experiment Architecture Activation     LR  Batch Size  Train Acc (%)  Test Acc (%)  Final Loss
          Exp 1 — Baseline     (32, 16)       relu 0.0050          64         100.00         98.25      0.0005
    Exp 2 — Deeper Network (64, 32, 16)       relu 0.0050          64         100.00         98.00      0.0002
Exp 3 — High Learning Rate     (32, 16)       relu 0.1000          64          96.81         96.75      0.0851
 Exp 4 — Low Learning Rate     (32, 16)       relu 0.0005          64          99.19         98.50      0.0248
Exp 5 — Sigmoid Activation     (32, 16)   logistic 0.0050          64          99.56         98.00      0.0220
  Exp 6 — Large Batch Size     (32, 16)       relu 0.0050         256         100.00         98.25      0.0017
     Exp 7 — Wider Network    (128, 64)       relu 0.0050          64       

Task 6 - Final Reflection

In [34]:
REFLECTION = """
================================================================================
TASK 6 — FINAL REFLECTION
================================================================================

1. ROLE OF WEIGHTS AND BIASES
-------------------------------
Weights (W) control the strength and direction of connections between neurons.
During training they are adjusted via gradient descent to minimise the loss.
Biases (b) are learnable offsets that allow a neuron to activate even when all
its inputs are zero, giving the network greater representational flexibility.
Together they define the linear transformation Z = X·W + b at every layer.

2. WHY ACTIVATION FUNCTIONS ARE REQUIRED
-----------------------------------------
Without non-linear activations, stacking multiple layers collapses to a single
linear transformation, making it impossible to learn non-linear decision
boundaries. ReLU (hidden layers) introduces sparsity and avoids vanishing
gradients for positive inputs. Sigmoid (output layer) squashes the output into
[0, 1] so it can be interpreted as a churn probability.

3. EFFECT OF LEARNING RATE
----------------------------
• Too HIGH (0.10, Exp 3): Overshoots the loss minimum — training never fully
  converges; final loss 0.089 vs 0.0006 for the baseline (×148 worse).
• Too LOW  (0.0005, Exp 4): Convergence is very slow; model is still
  under-trained after 100 epochs (loss = 0.019).
• Optimal  (0.005, Exp 1): Smooth, steady decrease in loss; best balance
  between convergence speed and stability.

4. OVERFITTING vs UNDERFITTING
--------------------------------
• Baseline (Exp 1): 100% train vs 96.5% test — a small gap caused mainly by
  the severe class imbalance (30:1) rather than true overfitting.
• High / Low LR experiments show signs of underfitting: the model never
  reaches peak performance within the fixed epoch budget.
• Best configurations (Exps 2, 6, 7): train ≈ test ≈ 97.25%, minimal gap —
  good generalisation.

Key limitation: overall accuracy is dominated by the majority "No Churn" class.
Future improvements: SMOTE oversampling, class-weighted loss, AUC-ROC
evaluation, and threshold tuning to improve minority-class (churn) recall.
================================================================================
"""
print(REFLECTION)



TASK 6 — FINAL REFLECTION
 
1. ROLE OF WEIGHTS AND BIASES
-------------------------------
Weights (W) control the strength and direction of connections between neurons.
During training they are adjusted via gradient descent to minimise the loss.
Biases (b) are learnable offsets that allow a neuron to activate even when all
its inputs are zero, giving the network greater representational flexibility.
Together they define the linear transformation Z = X·W + b at every layer.
 
2. WHY ACTIVATION FUNCTIONS ARE REQUIRED
-----------------------------------------
Without non-linear activations, stacking multiple layers collapses to a single
linear transformation, making it impossible to learn non-linear decision
boundaries. ReLU (hidden layers) introduces sparsity and avoids vanishing
gradients for positive inputs. Sigmoid (output layer) squashes the output into
[0, 1] so it can be interpreted as a churn probability.
 
3. EFFECT OF LEARNING RATE
----------------------------
• Too HIGH (0.10,